# Story 3 — The Geographic Heatmap
**Goal:** Identify which Brazilian states have the highest percentage of late deliveries and determine whether remote/northern states are disproportionately affected.

**Input:** `../data/02_delivered.parquet`  
**Output:** `../data/03_state_stats.parquet` + charts saved to `../exports/`

---

In [ ]:
import json
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

COLOR_MAP  = {'On Time': '#2ecc71', 'Late': '#f39c12', 'Super Late': '#e74c3c'}
print('Libraries loaded.')

## Step 1 — Load delivered orders

In [ ]:
delivered = pd.read_parquet('../data/02_delivered.parquet')
print(f'Loaded: {len(delivered):,} delivered orders across {delivered["customer_state"].nunique()} states')

## Step 2 — Aggregate by state

In [ ]:
state_stats = (
    delivered
    .groupby('customer_state')
    .agg(
        total_orders  = ('order_id', 'count'),
        late_orders   = ('delivery_status', lambda x: (x != 'On Time').sum()),
        avg_delay_days= ('delivery_delay_days', 'mean'),
        avg_review    = ('review_score', 'mean'),
    )
    .reset_index()
)
state_stats['pct_late']       = (state_stats['late_orders'] / state_stats['total_orders'] * 100).round(1)
state_stats['avg_delay_days'] = state_stats['avg_delay_days'].round(1)
state_stats['avg_review']     = state_stats['avg_review'].round(2)
state_stats = state_stats.sort_values('pct_late', ascending=False)

print('Top 10 states by % late deliveries:')
print(state_stats[['customer_state','total_orders','pct_late','avg_delay_days','avg_review']].head(10).to_string(index=False))

## Step 3 — Bar chart: % late by state

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
palette = ['#e74c3c' if p > 20 else '#f39c12' if p > 10 else '#2ecc71'
           for p in state_stats['pct_late']]
bars = ax.barh(state_stats['customer_state'], state_stats['pct_late'],
               color=palette, edgecolor='white')
ax.set_xlabel('% Late Deliveries', fontsize=12)
ax.set_title('Late Delivery Rate by Brazilian State', fontsize=15, fontweight='bold')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, state_stats['pct_late']):
    ax.text(val + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../exports/fig_state_late_rate.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Choropleth map of Brazil

In [ ]:
geojson_url = (
    'https://raw.githubusercontent.com/codeforamerica/click_that_hood/'
    'master/public/data/brazil-states.geojson'
)
with urllib.request.urlopen(geojson_url) as resp:
    brazil_geo = json.load(resp)

for feat in brazil_geo['features']:
    feat['id'] = feat['properties'].get('sigla', feat['properties'].get('uf', ''))

fig = px.choropleth(
    state_stats,
    geojson=brazil_geo,
    locations='customer_state',
    color='pct_late',
    color_continuous_scale='RdYlGn_r',
    range_color=(0, state_stats['pct_late'].max()),
    hover_data={'total_orders': True, 'avg_delay_days': True, 'avg_review': True},
    title='Late Delivery Rate (%) by Brazilian State',
    labels={'pct_late': '% Late', 'customer_state': 'State',
            'total_orders': 'Total Orders', 'avg_delay_days': 'Avg Delay (d)', 'avg_review': 'Avg Review'},
    fitbounds='locations',
    basemap_visible=False,
)
fig.update_layout(margin={'r': 0, 't': 40, 'l': 0, 'b': 0})
fig.show()

## Step 5 — Remote vs. Core state comparison
**Hypothesis:** Northern/remote states (far from São Paulo distribution hubs) are disproportionately late.

In [ ]:
remote_states = ['AM', 'RR', 'AC', 'AP', 'RO', 'PA', 'TO', 'MA']
core_states   = ['SP', 'RJ', 'MG', 'PR', 'RS', 'SC']

remote_pct = state_stats[state_stats['customer_state'].isin(remote_states)]['pct_late'].mean()
core_pct   = state_stats[state_stats['customer_state'].isin(core_states)]['pct_late'].mean()

print('─' * 50)
print(f' Remote/Northern states avg late rate:  {remote_pct:.1f}%')
print(f' Core/Southern states avg late rate:    {core_pct:.1f}%')
print(f' Remote states are {remote_pct/core_pct:.1f}× more likely to be late')
print('─' * 50)

# Side-by-side comparison bar
compare = pd.DataFrame({
    'Region': ['Remote/Northern', 'Core/Southern'],
    'Avg % Late': [round(remote_pct, 1), round(core_pct, 1)]
})
fig = px.bar(compare, x='Region', y='Avg % Late',
             color='Region', color_discrete_sequence=['#e74c3c', '#2ecc71'],
             text_auto='.1f',
             title='Remote vs Core States — Average Late Delivery Rate',
             labels={'Avg % Late': '% Late Deliveries'})
fig.update_traces(texttemplate='%{y:.1f}%', textposition='outside')
fig.update_layout(showlegend=False, yaxis_range=[0, max(remote_pct, core_pct) * 1.3])
fig.show()

## Step 6 — Save state stats

In [ ]:
state_stats.to_parquet('../data/03_state_stats.parquet', index=False)
print('Saved → ../data/03_state_stats.parquet')
print('\nNext: run 04_sentiment_correlation.ipynb')